# Comparer 7 modes d'orchestration — ou comment un budget invente un classement

Ce notebook n'enseigne pas un **axe** (une capacité) : il enseigne un **instrument** — comment on arbitre entre des architectures qui font toutes « la même chose ».

`scripts/compare_orchestration_modes.py` fait tourner **7 modes** sur le même texte et émet des métriques d'arbitrage (termine / wall-time / décide / périmètre). Sept modes, une seule mesure commune — et c'est là que le piège se referme.

**Le piège.** Au budget par défaut (**180 s**), 4 des 7 modes sont **tués** par le filet de sécurité, un 5ᵉ s'arrête de lui-même à 180 s sur 1/3 de ses phases — et **se déclare satisfait**. Le tableau produit reste **lisible, plausible, et trompeur** : ses trois colonnes donnent **trois chiffres différents pour la même exécution** — `Terminates` ✅ pour **7/7** (y compris les 4 modes tués : « termine » veut dire « a rendu un artefact », pas « a fini »), `success` pour **3/7**, et **2/7** seulement ont accompli toutes leurs phases. Il ne classe pas les modes par leurs propriétés, il les classe par *qui meurt le plus tard*. Ce notebook montre les **deux** runs côte à côte — sous-budget et calibré — pour que le lecteur voie que la mesure dépend de l'instrument, et qu'un ✅ n'est pas un compte d'achèvement.

**Corpus-free.** Aucun corpus chiffré, aucun `raw_text`, aucune clé API : les textes sont les **textes synthétiques** embarqués dans le script (aucun nom d'événement, de locuteur ni de date réels), et le notebook rejoue les mesures committées.


In [1]:
# Imports + provenance des deux runs. Aucune clé API n'est lue : les mesures sont committées.
import json
import subprocess
import sys

EXAMPLES = json.load(
    open("docs/coursia_contrib/orchestration_modes_compared_examples.json", encoding="utf-8")
)

PROV = EXAMPLES["provenance"]
print("Instrument :", PROV["instrument"])
print("Modèle     :", PROV["model"], "(BYOK — coût réel mesuré, pas estimé)")
print("Corpus     :", ", ".join(PROV["corpora"]), "—", PROV["corpus_note"])
print()
for run in PROV["runs"]:
    print(
        f"run {run['name']:12s} budget={run['max_wall_seconds']:>4}s "
        f"wall={run['wall_minutes']:>5.1f} min  coût=${run['cost_usd']:.4f}"
    )

UNDER = EXAMPLES["results"]["under_budget"]
CALIBRATED = EXAMPLES["results"]["calibrated"]
MODES = [r["mode"] for r in UNDER]
assert len(MODES) == 7, f"7 modes attendus, {len(MODES)} lus"
assert MODES == [r["mode"] for r in CALIBRATED], "les deux runs ne portent pas les mêmes modes"
print()
print("7 modes comparables lus dans les deux runs :", ", ".join(MODES))


Instrument : scripts/compare_orchestration_modes.py
Modèle     : gpt-5.6-luna (BYOK — coût réel mesuré, pas estimé)
Corpus     : corpus_A — textes synthétiques embarqués dans le script (aucun corpus chiffré, aucun raw_text)

run under_budget budget= 180s wall= 17.5 min  coût=$0.5504
run calibrated   budget= 600s wall= 35.9 min  coût=$0.9485

7 modes comparables lus dans les deux runs : pipeline_standard, pipeline_light, pipeline_full, conversational, conversation_deterministic, hierarchical_bridge, hierarchical_delegation


## 1. Le run au budget par défaut (180 s) — le tableau qui a l'air juste

Chaque mode reçoit **le même** budget. Trois colonnes, trois choses différentes, et c'est leur écart qui est le sujet :

- **`terminated_by_budget`** — le filet de sécurité du harnais a **tué** le mode. C'est le seul arrêt que le harnais décide.
- **`success`** — ce que le **mode** déclare de lui-même. Un mode peut se déclarer satisfait d'un travail partiel : son plafond interne (ici 180 s pour `conversational`) n'est pas le filet du harnais.
- **`phases_completed / phases_total`** — ce qui a réellement été fait.

Un tableau qui ne montre que `success` (ou un ✅) **surestime l'achèvement** : c'est exactement le piège de ce notebook.

**Le cas mesuré qui l'explique** — `conversational` s'arrête à **180,01 s sur 1/3** de ses phases, avec `success=True` **et** `terminated_by_budget=False`. Ce n'est pas une anomalie, et c'est vérifiable au source : son plafond est appliqué **à l'intérieur** du mode (sortie propre entre deux tours → verdict partiel **réel**, anti-#1019), tandis que le filet externe du harnais ne rattrape qu'un aller-retour LLM en vol — son délai (216 s pour un budget de 180 s) est délibérément **au-dessus** du plafond interne, précisément pour que le plafond interne tire le premier. Le mode sort donc de lui-même, honnêtement partiel. La leçon n'est pas « le rapport ment » : c'est que `success` répond à « le mode assume-t-il son verdict ? », pas à « a-t-il tout fait ? ».


In [2]:
# Le tableau du run sous-budget, reconstruit depuis les mesures — puis l'assertion qui compte.
def table(rows, cols):
    head = "| " + " | ".join(cols) + " |"
    sep = "|" + "|".join("---" for _ in cols) + "|"
    body = ["| " + " | ".join(str(fn(r)) for fn in vals) + " |" for r, vals in rows]
    print("\n".join([head, sep] + body))


def verdict(r):
    """Ce que le rapport dit du mode — en gardant visibles les trois registres."""
    if r["terminated_by_budget"]:
        return "⛔ tué par le harnais"
    if r["success"] and r["phases_completed"] == r["phases_total"]:
        return "✅ phases complètes"
    if r["success"]:
        return "⚠️ success=True, travail partiel"
    return "❌"


COLS = ["Mode", "Verdict", "Wall-time", "Phases", "success", "tué par le budget"]
rows = [
    (
        r,
        [
            lambda r: r["mode"],
            lambda r: verdict(r),
            lambda r: f"{r['duration_seconds']:.2f}s",
            lambda r: f"{r['phases_completed']}/{r['phases_total']}",
            lambda r: "✅" if r["success"] else "❌",
            lambda r: "⛔" if r["terminated_by_budget"] else "—",
        ],
    )
    for r in UNDER
]
print(f"### Budget {PROV['runs'][0]['max_wall_seconds']}s\n")
table(rows, COLS)

killed = [r["mode"] for r in UNDER if r["terminated_by_budget"]]
returned = [r["mode"] for r in UNDER if r["terminates"]]
claimed = [r["mode"] for r in UNDER if r["success"]]
complete = [r["mode"] for r in UNDER if r["phases_completed"] == r["phases_total"]]
print()
print(f"`terminates` (a rendu un artefact)     : {len(returned)}/7")
print(f"tués par le filet du harnais           : {len(killed)}/7 -> {', '.join(sorted(killed))}")
print(f"se déclarant success=True              : {len(claimed)}/7 -> {', '.join(sorted(claimed))}")
print(f"ayant fini TOUTES leurs phases         : {len(complete)}/7 -> {', '.join(sorted(complete))}")

partial = [r["mode"] for r in UNDER if r["success"] and r["phases_completed"] != r["phases_total"]]
print()
for mode in partial:
    r = next(x for x in UNDER if x["mode"] == mode)
    print(f"  ⚠️ {mode}: success=True sur {r['phases_completed']}/{r['phases_total']} phases, "
          f"{r['duration_seconds']:.2f}s, terminated_by_budget={r['terminated_by_budget']}")
    print(f"     portée déclarée par le mode : {r['scope_of_work']}")

# L'écart entre « se déclare satisfait » et « a fini » est la thèse du notebook :
assert len(killed) >= 4, (
    f"seulement {len(killed)}/7 modes tués par le budget — le run sous-budget ne sature plus "
    f"le filet, donc le contraste sous-budget/calibré ne démontrerait plus rien. Re-mesurer."
)
assert len(claimed) > len(complete), (
    f"success=True sur {len(claimed)} modes mais {len(complete)} seulement ont fini leurs "
    f"phases : l'écart qui fait le sujet du notebook a disparu, re-mesurer avant de réutiliser "
    f"ce carnet."
)
print()
print(f"=> {len(returned)} modes « terminent », {len(claimed)} se déclarent satisfaits, "
      f"{len(complete)} seulement ont fini : un ✅ n'est pas un compte d'achèvement.")


### Budget 180s

| Mode | Verdict | Wall-time | Phases | success | tué par le budget |
|---|---|---|---|---|---|
| pipeline_standard | ⛔ tué par le harnais | 180.01s | 6/15 | ❌ | ⛔ |
| pipeline_light | ✅ phases complètes | 148.76s | 3/3 | ✅ | — |
| pipeline_full | ⛔ tué par le harnais | 180.02s | 7/17 | ❌ | ⛔ |
| conversational | ⚠️ success=True, travail partiel | 180.01s | 1/3 | ✅ | — |
| conversation_deterministic | ✅ phases complètes | 0.06s | 3/3 | ✅ | — |
| hierarchical_bridge | ⛔ tué par le harnais | 180.01s | 2/4 | ❌ | ⛔ |
| hierarchical_delegation | ⛔ tué par le harnais | 180.02s | 1/5 | ❌ | ⛔ |

`terminates` (a rendu un artefact)     : 7/7
tués par le filet du harnais           : 4/7 -> hierarchical_bridge, hierarchical_delegation, pipeline_full, pipeline_standard
se déclarant success=True              : 3/7 -> conversation_deterministic, conversational, pipeline_light
ayant fini TOUTES leurs phases         : 2/7 -> conversation_deterministic, pipeline_light

  ⚠️ conversation

## 2. Pourquoi le classement est faux — les modes ne partagent pas le même axe de profondeur

Un mode « plus long » n'est pas un mode « plus faible » : les 7 modes ne mesurent pas la même chose. L'introspection ci-dessous est **déterministe** (aucun LLM, aucune JVM, aucun budget) — elle lit la structure des workflows.

**Le facteur 100.** `pipeline_light` finit en ~150 s ; `conversation_deterministic` en 0,06 s. Les deux sont « terminés » au sens du tableau. Comparer leurs durées n'a aucun sens : l'un fait 3 phases de DAG, l'autre un dialogue simulé.


In [3]:
# Introspection déterministe : le même script, sans run ni LLM ni JVM.
DEPTH = subprocess.run(
    [sys.executable, "scripts/compare_orchestration_modes.py", "--depth-parity"],
    capture_output=True,
    text=True,
    encoding="utf-8",
    timeout=300,
)
lines = [ln for ln in DEPTH.stdout.splitlines() if ln.startswith(("#", "|", "**"))]
print("\n".join(lines[:22]))
assert "pipeline_standard" in DEPTH.stdout and "dialogue-depth" in DEPTH.stdout, (
    "section depth-parity introuvable — l'instrument a changé, relire la sortie brute"
)
dims = [d for d in ("breadth", "delegation", "dialogue-depth") if d in DEPTH.stdout]
print()
print("axes de profondeur en présence :", ", ".join(dims))
assert len(dims) == 3, f"3 axes attendus, {len(dims)} trouvés"
print("=> les modes occupent 3 axes différents : leurs comptes ne sont pas comparables entre axes.")


## Depth-Parity Trade-off (C3 #1500)
| Mode | Depth dimension | Count | Nature |
|------|-----------------|-------|--------|
| pipeline_light | workflow phases (DAG) | 3 | breadth |
| pipeline_standard | workflow phases (DAG) | 15 | breadth |
| pipeline_full | workflow phases (DAG) | 17 | breadth |
| hierarchical_bridge | strategic objectives (default axes) | 4 | delegation |
| hierarchical_delegation | strategic objectives (LLM-derived, measured) | 4–4 objectives -> 5–5 tasks (n=3, corpus_A/B/C, firsthand R711, po-2023 projet-is) | delegation (3-tier depth) |
| conversational | dialogue macro-phases (multi-turn) | 3 | dialogue-depth |
| conversation_deterministic | dialogue macro-phases (deterministic) | 3 | dialogue-depth (no LLM) |

axes de profondeur en présence : breadth, delegation, dialogue-depth
=> les modes occupent 3 axes différents : leurs comptes ne sont pas comparables entre axes.


## 3. Le run calibré — mêmes modes, même texte, budget dérivé de la mesure

Le budget calibré n'est pas choisi au confort : le dépôt documente qu'à **180 s** `pipeline_standard` meurt à **6/15** de ses phases et qu'il lui faut **~500 s** pour atteindre 15/15 sur un corpus court (`CLAUDE.md`). On prend donc **600 s** — au-dessus du besoin documenté, avec une marge. C'est la règle que le notebook enseigne : **calibrer l'instrument avant de comparer**, sinon on compare des timeouts.

Seul le budget change. Le modèle, le corpus et les modes sont identiques — sinon le contraste ne prouverait rien.


In [4]:
# Le run calibré, et le contraste mode par mode.
rows = [
    (
        r,
        [
            lambda r: r["mode"],
            lambda r: verdict(r),
            lambda r: f"{r['duration_seconds']:.2f}s",
            lambda r: f"{r['phases_completed']}/{r['phases_total']}",
            lambda r: "✅" if r["success"] else "❌",
            lambda r: "⛔" if r["terminated_by_budget"] else "—",
        ],
    )
    for r in CALIBRATED
]
print(f"### Budget {PROV['runs'][1]['max_wall_seconds']}s\n")
table(rows, COLS)

by_mode_under = {r["mode"]: r for r in UNDER}
by_mode_cal = {r["mode"]: r for r in CALIBRATED}
print()
print("### Le contraste qui est le sujet du notebook\n")
print("| Mode | Phases @180s | Phases @600s | Verdict @180s | Verdict @600s |")
print("|------|--------------|--------------|---------------|---------------|")
for m in MODES:
    u, c = by_mode_under[m], by_mode_cal[m]
    print(
        f"| {m} | {u['phases_completed']}/{u['phases_total']} | "
        f"{c['phases_completed']}/{c['phases_total']} | {verdict(u)} | {verdict(c)} |"
    )

gained = [m for m in MODES if by_mode_cal[m]["phases_completed"] > by_mode_under[m]["phases_completed"]]
still = [m for m in MODES if by_mode_cal[m]["terminated_by_budget"]]
print()
print(f"modes ayant GAGNÉ des phases avec le budget calibré : {len(gained)}/7 -> {', '.join(gained) or 'aucun'}")
print(f"modes encore tués au budget calibré                 : {len(still)}/7 -> {', '.join(still) or 'aucun'}")
print()
print("=> la même exécution, une seule variable changée (le budget), et un classement différent.")


### Budget 600s

| Mode | Verdict | Wall-time | Phases | success | tué par le budget |
|---|---|---|---|---|---|
| pipeline_standard | ✅ phases complètes | 461.36s | 15/15 | ✅ | — |
| pipeline_light | ✅ phases complètes | 178.80s | 3/3 | ✅ | — |
| pipeline_full | ✅ phases complètes | 448.12s | 17/17 | ✅ | — |
| conversational | ⚠️ success=True, travail partiel | 600.02s | 2/3 | ✅ | — |
| conversation_deterministic | ✅ phases complètes | 0.06s | 3/3 | ✅ | — |
| hierarchical_bridge | ✅ phases complètes | 236.38s | 4/4 | ✅ | — |
| hierarchical_delegation | ⚠️ success=True, travail partiel | 217.02s | 3/5 | ✅ | — |

### Le contraste qui est le sujet du notebook

| Mode | Phases @180s | Phases @600s | Verdict @180s | Verdict @600s |
|------|--------------|--------------|---------------|---------------|
| pipeline_standard | 6/15 | 15/15 | ⛔ tué par le harnais | ✅ phases complètes |
| pipeline_light | 3/3 | 3/3 | ✅ phases complètes | ✅ phases complètes |
| pipeline_full | 7/17 | 17/17 | ⛔ tu

## À retenir

1. **Un budget par défaut n'est pas un budget neutre.** Il fabrique un classement (« qui meurt le plus tard ») qui ressemble à un résultat d'architecture. Les deux runs ci-dessus ne diffèrent que par cette variable.
2. **Calibrer avant de comparer.** Le budget calibré est *dérivé* de la mesure précédente, pas choisi au confort — et le notebook doit montrer le run sous-budget pour que l'étudiant voie la dépendance.
3. **Les modes n'occupent pas le même axe.** `pipeline` = largeur (beaucoup de capacités, peu profondes), `hierarchical` = délégation (peu d'objectifs, plusieurs étages), `conversational` = profondeur de dialogue. Le tableau « Termine / Wall-time » les aligne par interface, jamais par périmètre de travail — et cette asymétrie est un **choix**, pas un défaut (#1019).
4. **Un témoin n'est pas une référence.** `conversation_deterministic` ne consomme aucun LLM : c'est ce qui rend ce notebook exécutable sans clé, et c'est un **témoin déterministe** — pas « le mode de référence ».
5. **`success` est une déclaration du mode, pas un compte d'achèvement.** Le rapport tient plusieurs registres séparés — `terminates` (le mode a rendu un artefact), `terminated_by_budget` (le filet du harnais l'a tué), `success` (le mode assume son verdict), `phases_completed/total` (ce qui a été fait) — et c'est leur **écart** qui informe : **7 « terminent » / 4 tués / 3 `success` / 2 complets**, la même exécution. `conversational` sort à 1/3 de ses phases avec `success=True` parce que son plafond est appliqué **dans** le mode : une sortie propre entre deux tours rend un verdict partiel **réel** (anti-#1019), et le filet externe — volontairement plus tardif — n'a pas à tirer. C'est un partiel honnête, pas un succès dégradé ; mais lire son ✅ comme celui de `pipeline_light` (3/3) compare deux choses différentes. Et `Terminates ✅` sur les 4 modes **tués** montre que « termine » ne veut pas dire « a fini ». Ne jamais réduire ces registres à un ✅.
